<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-02/AI_Modul_2.7_Praktikum_Perulangan_For_While.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 2.7: Praktikum Perulangan (for, while)
### Arsitektur Iterasi Komputasional, Protokol Iterator, Kontrol Alur Eksekusi, dan Pemrosesan Aliran Telemetri Perkebunan

---

> **Diktat Terkait:** [AI_Modul_2.7_Perulangan_For_While.md](../../docs/part-02/AI_Modul_2.7_Perulangan_For_While.md)  
> **Outputs:** Script iterasi terstruktur (`for`, `while`), modul pemrosesan data deret waktu berbasis jendela geser (*sliding window moving average*), dan filter anomali menggunakan pernyataan kontrol (`break`, `continue`, `else`).  
> **Outcomes:** Mahasiswa terampil memilih pola perulangan yang efisien, menguasai protokol iterator Python, mengendalikan alur iterasi secara defensif, dan mencegah bahaya loop tak berhingga (*infinite loop*) serta galat pergeseran batas (*off-by-one errors*).  
> **Impacts:** Terciptanya sistem otomasi pemrosesan *Big Data* sensor perkebunan kelapa sawit yang tangguh, stabil pada perangkat komputasi tepi (*Edge IoT*), dan hemat konsumsi daya.
>
> **Tujuan Praktikum:**  
> 1. Mempraktikkan iterasi terhitung menggunakan `for`, `range`, `enumerate`, dan `zip`.  
> 2. Menginspeksi cara kerja protokol iterator tingkat rendah (`iter()` dan `next()`).  
> 3. Memahami dan menguji perilaku pernyataan kontrol `break`, `continue`, serta klausa `else` pada loop.  
> 4. Membangun pipeline pengendali irigasi cerdas berbasis *Sliding Window Moving Average*.  
> 5. Menyelesaikan 3 tantangan pemrograman: agregator curah hujan, simulasi konvergensi infiltrasi air, dan detektor hotspot kanopi drone berbasis `collections.deque`.

## 1. Persiapan Environment & Pemuatan Library
Jalankan sel berikut untuk memuat modul matematika, struktur data antrean berbatas, dan konfigurasi grafik visualisasi:

In [ ]:
# Mengimpor modul pustaka standar
import math
import sys
import time
from collections import deque
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional

# Mengimpor modul saintifik dan visualisasi grafik
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Konfigurasi gaya visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("[OK] Seluruh pustaka berhasil dimuat.")
print(f"[INFO] Versi Python Aktif: {sys.version.split()[0]}")

## 2. Iterasi Terhitung (`for`, `range`, `enumerate`, `zip`) & Protokol Iterator
Mengeksplorasi pembuatan deret aritmetika yang hemat memori, enumerasi berindeks, agregasi multi-koleksi secara paralel, serta pembedahan protokol `iter()` dan `next()`.

In [ ]:
# 1. Pemanfaatan enumerate() untuk melacak indeks sampel stasiun cuaca
suhu_harian = [26.5, 27.2, 29.0, 31.5, 28.4, 25.8, 26.0]
print("=== 1. DEMONSTRASI ENUMERATE() ===")
for hari, suhu in enumerate(suhu_harian, start=1):
    print(f"Hari ke-{hari}: Suhu Terukur = {suhu} C")

# 2. Pemanfaatan zip() untuk agregasi telemetri multi-sensor secara paralel
kelembaban_rh = [85.0, 82.0, 70.0, 62.0, 78.0, 90.0, 88.0]
print("\n=== 2. DEMONSTRASI ZIP() TELEMETRI IKLIM MIKRO ===")
for t, (suhu, rh) in enumerate(zip(suhu_harian, kelembaban_rh), start=1):
    # Estimasi sederhana Vapor Pressure Deficit (VPD) aproksimasi
    es = 0.61078 * math.exp((17.27 * suhu) / (suhu + 237.3))
    ea = es * (rh / 100.0)
    vpd = es - ea
    print(f"Hari #{t} | Suhu: {suhu:>4.1f} C | RH: {rh:>4.1f}% | VPD: {vpd:.3f} kPa")

# 3. Eksplorasi Protokol Iterator Tingkat Rendah (__iter__ dan __next__)
print("\n=== 3. DEKONSTRUKSI PROTOKOL ITERATOR PYTHON ===")
daftar_blok = ["BLOK-A01", "BLOK-A02", "BLOK-A03"]
iterator_manual = iter(daftar_blok)

print(f"Objek Iterator: {iterator_manual}")
print(f"Panggilan ke-1: {next(iterator_manual)}")
print(f"Panggilan ke-2: {next(iterator_manual)}")
print(f"Panggilan ke-3: {next(iterator_manual)}")

try:
    next(iterator_manual)
except StopIteration:
    print("Panggilan ke-4: [StopIteration Tertangkap! Iterasi Berhenti Alami]")

## 3. Kontrol Alur Iterasi: `break`, `continue`, dan Klausa `else` pada Loop
Memahami pemutus interupsi (`break`), penyaring derau (`continue`), dan klausa `else` yang dieksekusi hanya saat loop selesai tanpa interupsi.

In [ ]:
# Demonstrasi Klausa 'else' pada Loop untuk Validasi Batch
batch_tbs_a = [12.5, 14.0, 18.2, 16.5, 20.0]  # Seluruh buah lolos uji (> 5 kg)
batch_tbs_b = [15.0, 19.5, 3.2, 17.0, 18.0]   # Ada buah pasir (3.2 kg < 5 kg)

def verifikasi_batch(batch_data: List[float], nama_batch: str) -> None:
    print(f"\nMemverifikasi {nama_batch}: {batch_data}")
    for idx, berat in enumerate(batch_data, start=1):
        if berat < 5.0:
            print(f"  [REJECT] Ditemukan buah afkir pada item #{idx} ({berat} kg < 5.0 kg)! DITOLAK.")
            break
    else:
        # Hanya berjalan jika tidak pernah menyentuh 'break'
        print(f"  [APPROVED] {nama_batch} Lolos verifikasi! Seluruh tandan memenuhi standar mutu PKS.")

verifikasi_batch(batch_tbs_a, "BATCH SAWITA (GRADE A)")
verifikasi_batch(batch_tbs_b, "BATCH SAWIT B (CAMPURAN)")

# Demonstrasi 'continue' untuk Pembersihan Data Sensor
raw_sensor_readings = [28.5, None, 29.1, -999.0, 30.0, 29.8, None, 31.2]
data_bersih = []
print("\n=== FILTER DATA TELEMETRI MENGGUNAKAN 'continue' ===")
for r in raw_sensor_readings:
    if r is None or r < 0.0:
        continue  # Lewati rekaman korup
    data_bersih.append(r)

print(f"Data Mentah : {raw_sensor_readings}")
print(f"Data Bersih : {data_bersih}")

## 4. Perulangan Bersyarat (`while loop`), Variabel Sentinel, & Proteksi Watchdog
Mengontrol proses pengisian cairan fertigasi dan melindungi sistem dari bahaya perulangan tak hingga (*infinite loop*).

In [ ]:
# Simulasi Pengisian Tangki Fertirigasi dengan Watchdog Timer
def isi_tangki_fertigasi(target_volume: float, debit_per_detik: float, batas_watchdog_detik: int = 100) -> Tuple[int, float, bool]:
    volume_aktual = 0.0
    detik = 0
    watchdog_terpicu = False
    
    # While loop bersyarat dengan variabel sentinel volume_aktual
    while volume_aktual < target_volume:
        detik += 1
        volume_aktual += debit_per_detik
        
        # Proteksi Watchdog Timer untuk mencegah infinite loop jika pompa macet (debit <= 0)
        if detik >= batas_watchdog_detik:
            watchdog_terpicu = True
            print(f"  [WATCHDOG ALARM] Waktu pengisian melampaui batas aman ({batas_watchdog_detik}s)! Matikan daya.")
            break
            
    return detik, round(volume_aktual, 2), watchdog_terpicu

print("=== UJI WHILE LOOP: KONDISI NORMAL ===")
t_norm, v_norm, alert_norm = isi_tangki_fertigasi(target_volume=80.0, debit_per_detik=10.0)
print(f"Hasil Normal: Waktu = {t_norm}s | Volume = {v_norm}L | Alert = {alert_norm}")

print("\n=== UJI WHILE LOOP: KONDISI ANOMALI (DEBIT SANGAT KECIL / POMPA MACET) ===")
t_fail, v_fail, alert_fail = isi_tangki_fertigasi(target_volume=500.0, debit_per_detik=0.01, batas_watchdog_detik=50)
print(f"Hasil Anomali: Waktu = {t_fail}s | Volume = {v_fail}L | Alert = {alert_fail}")

## 5. Implementasi Kasus Terintegrasi: Pengendali Irigasi Cerdas Jendela Geser
Implementasi kelas industri `SmartIrrigationController` dari Diktat Bagian 5 yang memproses deret waktu sensor kelembaban tanah menggunakan *sliding window moving average* dan interupsi darurat.

In [ ]:
@dataclass(frozen=True)
class PembacaanSensorTanah:
    timestamp_detik: int
    kelembaban_tanah_persen: float
    status_sensor_valid: bool

class SmartIrrigationController:
    def __init__(self, ukuran_jendela: int = 3, ambang_kering_persen: float = 32.0, ambang_kritis_tripwire: float = 15.0) -> None:
        self.k = ukuran_jendela
        self.ambang_kering = ambang_kering_persen
        self.ambang_tripwire = ambang_kritis_tripwire

    def proses_aliran_telemetri(self, aliran_data: List[PembacaanSensorTanah]) -> Dict[str, Any]:
        total_data = len(aliran_data)
        data_valid: List[float] = []
        log_peristiwa: List[str] = []
        tripwire_aktif = False

        # 1. Loop filtrasi data sensor dengan 'continue' dan 'break'
        for s in aliran_data:
            if not s.status_sensor_valid:
                log_peristiwa.append(f"[SKIP] T={s.timestamp_detik}s: Sensor status INVALID.")
                continue
            if not (0.0 <= s.kelembaban_tanah_persen <= 100.0):
                log_peristiwa.append(f"[SKIP] T={s.timestamp_detik}s: Nilai di luar jangkauan fisik {s.kelembaban_tanah_persen}%.")
                continue
                
            data_valid.append(s.kelembaban_tanah_persen)
            
            # Interupsi darurat jika kelembaban turun di bawah ambang batas kritis
            if s.kelembaban_tanah_persen < self.ambang_tripwire:
                log_peristiwa.append(f"[INTERUPSI KRITIS] T={s.timestamp_detik}s: {s.kelembaban_tanah_persen}% < {self.ambang_tripwire}%!")
                tripwire_aktif = True
                break
        else:
            log_peristiwa.append("[INFO] Batch tuntas diproses tanpa interupsi darurat.")

        # 2. Algoritma Jendela Geser (Sliding Window Moving Average)
        n = len(data_valid)
        hasil_ma: List[Tuple[int, float, str]] = []
        if n >= self.k:
            for i in range(n - self.k + 1):
                window = data_valid[i : i + self.k]
                ma = sum(window) / self.k
                mode = "POMPA ON" if ma < self.ambang_kering else "POMPA OFF"
                hasil_ma.append((i, round(ma, 2), mode))

        return {
            "total": total_data,
            "valid": n,
            "tripwire": tripwire_aktif,
            "moving_average": hasil_ma,
            "logs": log_peristiwa
        }

# Uji Coba Simulasi
dataset_telemetri = [
    PembacaanSensorTanah(10, 42.0, True),
    PembacaanSensorTanah(20, 38.5, True),
    PembacaanSensorTanah(30, 999.0, False),  # Korup
    PembacaanSensorTanah(40, 35.0, True),
    PembacaanSensorTanah(50, 30.5, True),
    PembacaanSensorTanah(60, 28.0, True),
    PembacaanSensorTanah(70, 26.2, True),
    PembacaanSensorTanah(80, 24.0, True),
    PembacaanSensorTanah(90, 11.5, True),   # Tripwire kritis (< 15%)
]

controller = SmartIrrigationController(ukuran_jendela=3, ambang_kering_persen=32.0, ambang_kritis_tripwire=15.0)
hasil_laporan = controller.proses_aliran_telemetri(dataset_telemetri)

print("=== HASIL AUDIT PIPELINE TELEMETRI IRIGASI ===")
print(f"Total Paket: {hasil_laporan['total']} | Valid: {hasil_laporan['valid']} | Tripwire Aktif: {hasil_laporan['tripwire']}")
for entry in hasil_laporan['moving_average']:
    print(f"  Jendela #{entry[0]} -> MA: {entry[1]}% | Status: {entry[2]}")

## 6. Solusi Tantangan Mandiri (Scaffolded Challenges)
Di bawah ini disajikan implementasi lengkap untuk ketiga tantangan praktikum berjenjang.

In [ ]:
# === TANTANGAN 1: AGREGATOR CURAH HUJAN BULANAN ===
def analisis_curah_hujan(data_harian: List[float]) -> Dict[str, Any]:
    """
    Menghitung akumulasi curah hujan bulanan, rata-rata, dan hari hujan lebat
    menggunakan perulangan 'for' dan 'enumerate'.
    """
    total_hujan = 0.0
    hari_hujan_lebat = []
    hari_dengan_hujan = 0
    
    for hari, curah in enumerate(data_harian, start=1):
        if curah <= 0.0:
            continue  # Hari tanpa hujan dilewati
            
        total_hujan += curah
        hari_dengan_hujan += 1
        
        if curah > 50.0:
            hari_hujan_lebat.append((hari, curah))
            
    rata_rata = total_hujan / len(data_harian) if data_harian else 0.0
    
    return {
        "total_akumulasi_mm": round(total_hujan, 2),
        "rata_rata_harian_mm": round(rata_rata, 2),
        "jumlah_hari_hujan": hari_dengan_hujan,
        "daftar_hari_hujan_lebat": hari_hujan_lebat
    }

# Data curah hujan simulasi 30 hari
np.random.seed(101)
hujan_simulasi = [0.0 if np.random.rand() > 0.4 else round(float(np.random.exponential(scale=18.0)), 1) for _ in range(30)]
hujan_simulasi[12] = 68.5  # Hari hujan ekstrem
hujan_simulasi[24] = 54.0  # Hari hujan lebat

res_hujan = analisis_curah_hujan(hujan_simulasi)
print("=== TANTANGAN 1: AUDIT CURAH HUJAN BULANAN ===")
for k, v in res_hujan.items():
    print(f"{k:<25}: {v}")

In [ ]:
# === TANTANGAN 2: SIMULASI KONVERGENSI INFILTRASI AIR (WHILE LOOP) ===
def simulasi_konvergensi_infiltrasi(debit_awal: float, batas_toleransi: float = 0.001) -> Tuple[int, float]:
    """
    Menghitung konvergensi persamaan laju peresapan air tanah menuju kondisi steady-state
    Q_{t+1} = 0.85 * Q_t + 0.15 * 2.0 menggunakan while loop dengan batas 500 iterasi.
    """
    q_saat_ini = debit_awal
    iterasi = 0
    maks_iterasi = 500
    
    while iterasi < maks_iterasi:
        iterasi += 1
        q_berikutnya = (0.85 * q_saat_ini) + (0.15 * 2.0)
        selisih = abs(q_berikutnya - q_saat_ini)
        
        if selisih < batas_toleransi:
            return iterasi, round(q_berikutnya, 4)
            
        q_saat_ini = q_berikutnya
        
    return iterasi, round(q_saat_ini, 4)

print("\n=== TANTANGAN 2: KONVERGENSI LAJU INFILTRASI ===")
langkah_konv, nilai_tunak = simulasi_konvergensi_infiltrasi(debit_awal=12.5, batas_toleransi=0.0005)
print(f"Konvergensi Tercapai pada Iterasi ke-{langkah_konv} dengan Nilai Steady-State Q = {nilai_tunak} L/s")

In [ ]:
# === TANTANGAN 3: DETEKTOR HOTSPOT KANOPI SAWIT DENGAN DEQUE ===
class CanopyHotspotDetector:
    """
    Detektor titik panas kanopi sawit drone termal berbasis sliding window deque(maxlen=5).
    Hotspot terpicu jika: suhu_saat_ini > (MA + 3.0 C).
    """
    def __init__(self, ambang_kenaikan_c: float = 3.0) -> None:
        self.ambang = ambang_kenaikan_c
        self.jendela_suhu = deque(maxlen=5)
        self.daftar_hotspot: List[Tuple[int, float, float]] = [] # (index, suhu, ma)

    def proses_titik(self, t: int, suhu: float) -> Tuple[Optional[float], bool]:
        if len(self.jendela_suhu) < 5:
            self.jendela_suhu.append(suhu)
            return None, False
            
        ma = sum(self.jendela_suhu) / len(self.jendela_suhu)
        is_hotspot = suhu > (ma + self.ambang)
        if is_hotspot:
            self.daftar_hotspot.append((t, suhu, ma))
            
        self.jendela_suhu.append(suhu) # Deque otomatis membuang elemen tertua O(1)
        return round(ma, 2), is_hotspot

# Simulasi Aliran Suhu Kanopi Drone (60 Detik Penerbangan)
np.random.seed(42)
suhu_kanopi = np.random.normal(loc=31.0, scale=0.8, size=60).tolist()
# Sisipkan hotspot kebakaran lahan kecil pada detik ke-28 dan 48
suhu_kanopi[28] = 36.8
suhu_kanopi[48] = 38.5

detector = CanopyHotspotDetector(ambang_kenaikan_c=3.0)
ma_list = []

for t, s in enumerate(suhu_kanopi):
    ma_val, flag = detector.proses_titik(t, s)
    ma_list.append(ma_val if ma_val is not None else s)

print("\n=== TANTANGAN 3: DETEKSI HOTSPOT KANOPI DRONE TERMAL ===")
print(f"Total Titik Hotspot Terdeteksi: {len(detector.daftar_hotspot)}")
for idx, val, ma in detector.daftar_hotspot:
    print(f"  [HOTSPOT DETECTED] Detik #{idx:02d} | Suhu: {val:.1f} C (MA Latar: {ma:.1f} C, Selisih: +{val-ma:.1f} C)")

# Visualisasi Grafik Telemetri Kanopi
plt.figure(figsize=(10, 5))
plt.plot(suhu_kanopi, color='royalblue', lw=1.5, label='Suhu Termal Aktual (C)')
plt.plot(ma_list, color='darkorange', lw=1.8, linestyle='--', label='Rata-rata Bergerak (MA k=5)')

for idx, val, _ in detector.daftar_hotspot:
    plt.scatter(idx, val, color='red', s=80, zorder=5, label='Peringatan Hotspot' if idx == detector.daftar_hotspot[0][0] else "")

plt.xlabel('Waktu Terbang Drone (Detik)', fontweight='bold')
plt.ylabel('Suhu Kanopi (C)', fontweight='bold')
plt.title('Deteksi Hotspot Kebakaran Lahan Menggunakan Jendela Geser Deque (k=5)', fontweight='bold', fontsize=12)
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('docs/assets/simulasi_deteksi_hotspot_kanopi.png', dpi=150)
print("[INFO] Grafik deteksi hotspot berhasil dirender ke 'docs/assets/simulasi_deteksi_hotspot_kanopi.png'.")